In [ ]:
# Imports
from google import genai
from google.genai import types
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import json
import time
import base64
import re
import openpyxl
import sys
import importlib
sys.path.append('..')
import common.prompts as prompts
import common.file_processing as file_processing
importlib.reload(prompts)
importlib.reload(file_processing)

In [ ]:
load_dotenv(override=True)

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

gemini_client = genai.Client()
openai_client = OpenAI()

gemini_model = os.getenv("GEMINI_MODEL")
openai_model = os.getenv("OPENAI_MODEL")

images_folder_path = os.getenv("IMG_PATH")
output_folder_path = os.getenv("OUTPUT_FOLDER_PATH")

In [ ]:
def parse_json(text):
    """Parse JSON from model response with better error handling"""
    if not text:
        raise ValueError("Input text is None or empty")
    
    if not isinstance(text, str):
        raise ValueError(f"Input text is not a string, got {type(text)}")
    
    # Clean the text
    text = text.strip()
    
    # Try direct parsing first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # Try to find JSON between first { and last }
    try:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and end > start:
            json_str = text[start:end+1]
            return json.loads(json_str)
    except json.JSONDecodeError:
        pass

    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, text, re.DOTALL)
    
    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue
    
    # If we get here, no valid JSON was found
    raise ValueError(f"No valid JSON found in text: {text[:200]}...")
    
def count_images(folder_path):
    """Count numbered images"""
    count = 0
    index = 1
    while os.path.exists(os.path.join(folder_path, f"{index}.jpg")):
        count += 1
        index += 1
    return count

In [ ]:
# Step 1: Gemini serving size estimate
def gemini_serving_size(image):
    """Gemini rough portion estimate"""
    try:
        response = gemini_client.models.generate_content(
            model=gemini_model,
            contents=[
                types.Part.from_bytes(
                    data=image,
                    mime_type='image/jpeg',
                ),
                prompts.build_serving_estimation_prompt()
            ],
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json",
                max_output_tokens=512,
            )
        )
        data = parse_json(response.text)
        return {
            'success': True,
            'description': data.get('description'),
            'serving_size': float(data.get('serving_size', 0)),
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}
    
# Step 2: GPT-4 nutrition analysis
def gpt_nutritionist(image, description, serving_size):
    try:
        nutrition_prompt = prompts.build_nutrition_estimation_prompt(
            description,
            serving_size
        )
        
        image_base64 = base64.b64encode(image).decode("utf-8")
        
        response = openai_client.chat.completions.create(
            model=openai_model,
            messages=[
            {
                "role": "system",
                "content": prompts.SYSTEM_PROMPT_NUTRITIONIST
            },  
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": nutrition_prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_base64}"
                        }
                    }
                ]
            }],
            max_tokens=1024,
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        data = parse_json(response.choices[0].message.content)
        return {
            'success': True,
            'description': data.get('description'),
            'calories': float(data.get('calories', 0)),
            'proteins': float(data.get('proteins', 0)),
            'carbohydrates': float(data.get('carbohydrates', 0)),
            'fats': float(data.get('fats', 0)),
            'serving_size': float(data.get('serving_size', 0))
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

In [ ]:
# Main chained analysis
def analyze_image_chained(image_path, index):
    """Complete chained analysis for one image"""
    start_time = time.time()
    file_name = os.path.basename(image_path)
    
    print(f"\n🔄 Processing {index}: {file_name}")
    with open(image_path, "rb") as f:
            image_byte = f.read()
    
    print("  Step 1: Serving size (Gemini)...")
    serving_size = gemini_serving_size(image_byte)
    if not serving_size['success']:
        return {'success': False, 'error': f"Step 1 failed: {serving_size['error']}", 'index': index}
    print(f"    → {serving_size['serving_size']}g - {serving_size['description'][:50]}...")
    
    time.sleep(2)  # Rate limit
    
    print("  Step 2: Nutrition analysis (GPT-4)...")
    nutrition = gpt_nutritionist(image_byte, serving_size['description'], serving_size['serving_size'])
    if not nutrition['success']:
        return {'success': False, 'error': f"Step 3 failed: {nutrition['error']}", 'index': index}
    print(f"    → {nutrition['proteins']}g - {nutrition['carbohydrates']}g - {nutrition['fats']}g...")
    
    total_time = time.time() - start_time
    print(f"  ✅ Complete! {nutrition['calories']} kcal in {total_time:.1f}s")
    
    return {
        'success': True,
        'index': index,
        'file_name': file_name,
        'serving_size': serving_size,
        'nutrition': nutrition,
        'processing_time': total_time
    }

In [ ]:
# Process dataset
def process_chained_dataset(file_name, folder_path, start=1, end=None):
    """Process images with chained analysis"""
    
    total_images = count_images(folder_path)
    if end is None:
        end = total_images
    end = min(end, total_images)
    
    print(f"🚀 Processing images {start} to {end} ({end-start+1} total)")
    
    results = []
    successful = 0
    
    for i in range(start, end + 1):
        image_path = os.path.join(folder_path, f"{i}.jpg")
        result = analyze_image_chained(image_path, i)
        results.append(result)
        
        if result['success']:
            successful += 1
    
    print(f"\n🎉 Completed! {successful}/{len(results)} successful")
    
    # Save results
    output_file = f"{file_name}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"📁 Results saved to: {output_file}")
    return results

In [ ]:
def export_to_excel(file_name):
    """Export results to Excel"""
    with open(f"{file_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Prepare final results
    final_results = []
    for item in data:
        if item['success']:
            nutrition = item['nutrition']
            final_results.append({
                'id': item['index'],
                'description': nutrition['description'],
                'serving_size': nutrition['serving_size'],
                'calories': nutrition['calories'],
                'proteins': nutrition['proteins'],
                'carbohydrates': nutrition['carbohydrates'],
                'fats': nutrition['fats']
            })
    
    # Create DataFrame and export
    df = pd.DataFrame(final_results)
    output_path = os.path.join(output_folder_path, f"{file_name}.xlsx")
    df.to_excel(output_path, index=False)
    
    print(f"✅ Excel exported: {output_path}")
    print(f"📊 {len(final_results)} successful analyses")
    
    return output_path

In [ ]:
file_name = "nutria_gemini_chained_gpt"

In [ ]:
results = process_chained_dataset(file_name, images_folder_path, start=1, end=count_images(images_folder_path))

In [ ]:
excel_path = export_to_excel(file_name)